# Electivo de Bioinformática — Clase 7

## Llamado de variantes, formato VCF y anotación de variantes. Interpretación

**Programa:** Doctorado — 2º año
**Duración:** 3 horas

### Objetivos de la clase

Al finalizar esta clase, serán capaces de:

1. Explicar, en términos simples, qué hace un **llamador de variantes** a partir de un BAM alineado.
2. Llamar variantes con **GATK4 HaplotypeCaller** sobre el BAM que dejamos listo la clase pasada.
3. Leer un archivo **VCF**: identificar sus columnas y los campos más importantes del genotipo (`GT`, `AD`, `DP`, `GQ`).
4. **Anotar** las variantes encontradas (en qué gen caen, qué tipo de cambio producen) con `bcftools csq`, e **interpretar** un resultado concreto.

Hoy no volvemos a instalar ni preparar lo de las clases anteriores: partimos del BAM final de la clase 6 y agregamos solo lo nuevo (un llamador de variantes y un anotador).

---
## 1. Dónde quedamos (5 min)

En la clase 6 dejamos, para la muestra `mtDNA_sample`, un BAM **mejorado**: alineado, con las parejas sincronizadas, los duplicados marcados y las calidades recalibradas (`~/bioinfo/clase6/resultados/mtDNA_sample.final.bam`).

Ese BAM es exactamente el insumo que necesita un llamador de variantes. Hoy damos el siguiente paso: a partir de ese BAM, encontrar en qué posiciones la muestra difiere de la referencia, y qué significa cada diferencia.

---
## 2. Verificar que el ambiente esté listo (15 min)

Necesitamos lo que ya quedó de las clases 5 y 6 —la referencia indexada, el BAM final de `mtDNA_sample` y el archivo de sitios conocidos— más **una herramienta nueva** para anotar variantes: `bcftools`.

Si guardaron estos archivos en otras rutas, ajusten las variables antes de correr la celda.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

CLASE5=~/bioinfo/clase5
CLASE6=~/bioinfo/clase6

echo "=== Herramientas en el ambiente 'mejoramiento' ==="
for HERR in gatk samtools bcftools; do
    if command -v "$HERR" >/dev/null 2>&1; then
        echo "OK    $HERR"
    else
        echo "FALTA $HERR"
    fi
done

echo
echo "=== Referencia (de la clase 5) ==="
REF="$CLASE5/referencia/chrM.fasta"
for ARCHIVO in "$REF" "${REF}.fai" "${REF%.*}.dict"; do
    if [ -s "$ARCHIVO" ]; then echo "OK    $ARCHIVO"; else echo "FALTA $ARCHIVO"; fi
done

echo
echo "=== BAM final y sitios conocidos (de las clases 5 y 6) ==="
for ARCHIVO in "$CLASE6/resultados/mtDNA_sample.final.bam" \
               "$CLASE6/resultados/mtDNA_sample.final.bai" \
               "$CLASE5/sitios_conocidos.vcf"; do
    if [ -s "$ARCHIVO" ]; then echo "OK    $ARCHIVO"; else echo "FALTA $ARCHIVO"; fi
done

Si falta **`bcftools`**, agréguenlo al ambiente `mejoramiento` (trae consigo `bgzip` y `tabix`, dos herramientas de `htslib` para comprimir e indexar archivos por posición que vamos a necesitar más adelante):

```bash
conda install -y -n mejoramiento -c conda-forge -c bioconda bcftools
```

Si falta algún archivo de la referencia, el BAM final o `sitios_conocidos.vcf`: revisen las clases 5 y 6 (secciones de verificación de esos notebooks) antes de seguir. Todo lo demás de hoy depende de que estos tres grupos de archivos existan.

---
## 3. ¿Qué es un llamado de variantes? (10 min)

Un **llamador de variantes** recorre el BAM posición por posición y, en cada una, mira qué dicen las lecturas que la cubren:

- Si, en general, todas las lecturas de buena calidad coinciden con la referencia, no hay nada que reportar.
- Si un grupo consistente de lecturas muestra una base distinta —y esa diferencia no se explica solo por errores aleatorios de secuenciación—, se reporta como una **variante candidata**, con una calidad (`QUAL`) que estima qué tan confiable es.

Distintos programas usan distintos modelos estadísticos para decidir qué es señal y qué es ruido; hoy usamos **GATK4 HaplotypeCaller**, uno de los más usados en clínica e investigación (es el mismo que ya instalamos en la clase 5 para BQSR). El resultado se guarda en un archivo **VCF** (*Variant Call Format*), el formato estándar para representar variantes.

---
## 4. Llamar variantes con GATK4 HaplotypeCaller (15 min)

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

mkdir -p ~/bioinfo/clase7/resultados
cd ~/bioinfo/clase7

REF=~/bioinfo/clase5/referencia/chrM.fasta
BAM=~/bioinfo/clase6/resultados/mtDNA_sample.final.bam

gatk HaplotypeCaller -R "$REF" -I "$BAM" -O resultados/mtDNA_sample.vcf

> Nota: el genoma mitocondrial no es diploide como los cromosomas nucleares —cada célula tiene muchas copias—. GATK asume por defecto un organismo diploide (como si hubiera como máximo dos versiones distintas por posición); para esta muestra simulada eso no cambia el resultado, pero en un análisis real de mtDNA se ajustaría la ploidía o se usaría una herramienta pensada para mitocondria.

Veamos qué encontró:

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase7
grep -v "^##" resultados/mtDNA_sample.vcf

`HaplotypeCaller` debería haber encontrado **11 variantes candidatas**. Compárenlo con lo que les salió a ustedes.

---
## 5. El formato VCF (20 min)

Un VCF tiene dos partes: líneas de **encabezado** (empiezan con `##`, definen qué significa cada campo) y una **tabla** con una fila por variante (la línea de nombres de columna empieza con un solo `#`).

Miremos el encabezado:

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase7
grep "^##" resultados/mtDNA_sample.vcf | grep -E "^##(FORMAT|fileformat)" 

Y las 8 columnas obligatorias, más `FORMAT` y la columna de la muestra:

| Columna | Qué contiene |
|---|---|
| `CHROM` | El cromosoma/contig (para nosotros, siempre `NC_012920.1`) |
| `POS` | La posición de la variante (1-based) |
| `ID` | Identificador en alguna base de datos (`.` si no hay) |
| `REF` | La base (o bases) de la referencia en esa posición |
| `ALT` | La base (o bases) que se observó en la muestra |
| `QUAL` | Qué tan confiable es la llamada (más alto = más confiable) |
| `FILTER` | Si pasó los filtros de calidad (`PASS`, o el nombre del filtro que falló) |
| `INFO` | Estadísticas agregadas de la variante (profundidad, etc.) |
| `FORMAT` | Qué campos trae la columna de la muestra, y en qué orden |
| `mtDNA_sample` | Los valores de esos campos para esta muestra |

Tomemos una fila real de su propio resultado —la posición 4827, un buen ejemplo porque es una variante donde **todas** las lecturas muestran el cambio— y desarmemos la columna `FORMAT`:

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase7
grep -v '^#' resultados/mtDNA_sample.vcf | awk -F'\t' '$2==4827'

En esa fila, `FORMAT` es `GT:AD:DP:GQ:PL` y la columna de la muestra trae un valor por cada uno, en el mismo orden:

| Campo | Significado | Valor en esta variante |
|---|---|---|
| `GT` | Genotipo: qué alelo(s) tiene la muestra. `0`=referencia, `1`=el primer `ALT`. `0/1` = heterocigoto, `1/1` = homocigoto para el alelo alternativo | `1/1` (todas las copias con el cambio) |
| `AD` | Cuántas lecturas apoyan cada alelo (referencia, alternativo) | `0,17` (0 lecturas con la referencia, 17 con el cambio) |
| `DP` | Profundidad total en esa posición | `17` |
| `GQ` | Confianza en el genotipo llamado (escala tipo Phred) | `51` |
| `PL` | Probabilidad (en escala Phred, invertida) de cada genotipo posible (`0/0`, `0/1`, `1/1`); el más bajo es el más probable | `588,51,0` → `1/1` es el más probable |

Comparen esta fila con la de la posición 591 (`0/1`, heterocigota): ahí `AD` va a repartirse entre referencia y alternativo, no va a ser todo hacia un lado.

---
## 6. ¿Qué tan bien nos fue? Comparar contra la respuesta conocida (15 min)

Acá viene un detalle que quedó pendiente desde la clase 4: cuando generamos el FASTQ con `dwgsim`, además de los datos crudos se guardó un archivo con las mutaciones **exactas** que introdujo el simulador —lo usamos como sitios conocidos para BQSR en la clase 5/6, sin mirarlo en detalle—. Como es una muestra **simulada**, ese archivo es la respuesta correcta: sabemos con certeza qué variantes tiene la muestra. Comparemos.

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase7

grep -v "^#" resultados/mtDNA_sample.vcf | cut -f2 | sort -n > resultados/llamadas.txt
grep -v "^#" ~/bioinfo/clase5/sitios_conocidos.vcf | cut -f2 | sort -n > resultados/conocidas.txt

echo "--- Encontradas por HaplotypeCaller Y en la respuesta conocida ---"
grep -Fxf resultados/conocidas.txt resultados/llamadas.txt || echo "(ninguna)"

echo
echo "--- En la respuesta conocida pero el llamador NO las encontro ---"
grep -Fxvf resultados/llamadas.txt resultados/conocidas.txt || echo "(ninguna)"

echo
echo "--- El llamador reporto una posicion que NO esta en la respuesta conocida ---"
grep -Fxvf resultados/conocidas.txt resultados/llamadas.txt || echo "(ninguna)"

Debería faltarles solo la posición **33**, y no debería sobrarles ninguna. Miremos por qué se escapó esa:

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase7
samtools depth -a -r NC_012920.1:25-45 ~/bioinfo/clase6/resultados/mtDNA_sample.final.bam

La cobertura ahí es de solo 1-2 lecturas: la posición 33 está cerca del extremo de nuestra referencia lineal. El genoma mitocondrial es en realidad **circular**, pero acá lo tratamos como una secuencia lineal, así que los primeros y los últimos ~100 pb tienen menos lecturas superpuestas que el resto del genoma —con tan poca cobertura, el llamador prefiere no arriesgarse—. En un análisis real esto se resuelve "rotando" la referencia o usando herramientas pensadas para genomas circulares; para esta clase basta con entender **por qué** pasó.

> **Para discutir (en parejas, 3 min):** con datos reales (no simulados) no tenemos una "respuesta correcta" para comparar. ¿Cómo se evalúa entonces la confiabilidad de un llamado de variantes?

---
## 7. Anotar las variantes: ¿en qué gen caen y qué producen? (25 min)

Saber que hay una variante en la posición 4827 no dice mucho por sí solo. **Anotar** una variante es responder: ¿en qué gen cae? ¿Es una región que codifica proteína? Si lo es, ¿el cambio altera el aminoácido?

Para esto usamos `bcftools csq`, que necesita —además del VCF y la referencia— un archivo con las coordenadas de los genes (formato **GFF3**). Ya lo preparamos con las coordenadas oficiales de los 37 genes del genoma mitocondrial humano (13 genes que codifican proteínas, 22 ARN de transferencia, 2 ARN ribosomal), tal como están publicadas en NCBI/MITOMAP —no hace falta que lo escriban ustedes—:

In [ ]:
%%bash
cd ~/bioinfo/clase7

cat > chrM.gff3 << 'EOF'
##gff-version 3
##sequence-region NC_012920.1 1 16569
NC_012920.1	rCRS	gene	577	647	.	+	.	ID=gene:MT-TF;Name=MT-TF;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	577	647	.	+	.	ID=transcript:MT-TF-1;Parent=gene:MT-TF;Name=MT-TF-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	577	647	.	+	.	ID=exon:MT-TF-1.1;Parent=transcript:MT-TF-1
NC_012920.1	rCRS	gene	648	1601	.	+	.	ID=gene:MT-RNR1;Name=MT-RNR1;biotype=Mt_rRNA
NC_012920.1	rCRS	mRNA	648	1601	.	+	.	ID=transcript:MT-RNR1-1;Parent=gene:MT-RNR1;Name=MT-RNR1-1;biotype=Mt_rRNA
NC_012920.1	rCRS	exon	648	1601	.	+	.	ID=exon:MT-RNR1-1.1;Parent=transcript:MT-RNR1-1
NC_012920.1	rCRS	gene	1602	1670	.	+	.	ID=gene:MT-TV;Name=MT-TV;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	1602	1670	.	+	.	ID=transcript:MT-TV-1;Parent=gene:MT-TV;Name=MT-TV-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	1602	1670	.	+	.	ID=exon:MT-TV-1.1;Parent=transcript:MT-TV-1
NC_012920.1	rCRS	gene	1671	3229	.	+	.	ID=gene:MT-RNR2;Name=MT-RNR2;biotype=Mt_rRNA
NC_012920.1	rCRS	mRNA	1671	3229	.	+	.	ID=transcript:MT-RNR2-1;Parent=gene:MT-RNR2;Name=MT-RNR2-1;biotype=Mt_rRNA
NC_012920.1	rCRS	exon	1671	3229	.	+	.	ID=exon:MT-RNR2-1.1;Parent=transcript:MT-RNR2-1
NC_012920.1	rCRS	gene	3230	3304	.	+	.	ID=gene:MT-TL1;Name=MT-TL1;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	3230	3304	.	+	.	ID=transcript:MT-TL1-1;Parent=gene:MT-TL1;Name=MT-TL1-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	3230	3304	.	+	.	ID=exon:MT-TL1-1.1;Parent=transcript:MT-TL1-1
NC_012920.1	rCRS	gene	3307	4262	.	+	.	ID=gene:MT-ND1;Name=MT-ND1;biotype=protein_coding
NC_012920.1	rCRS	mRNA	3307	4262	.	+	.	ID=transcript:MT-ND1-1;Parent=gene:MT-ND1;Name=MT-ND1-1;biotype=protein_coding
NC_012920.1	rCRS	exon	3307	4262	.	+	.	ID=exon:MT-ND1-1.1;Parent=transcript:MT-ND1-1
NC_012920.1	rCRS	CDS	3307	4262	.	+	0	ID=CDS:MT-ND1-1;Parent=transcript:MT-ND1-1
NC_012920.1	rCRS	gene	4263	4331	.	+	.	ID=gene:MT-TI;Name=MT-TI;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	4263	4331	.	+	.	ID=transcript:MT-TI-1;Parent=gene:MT-TI;Name=MT-TI-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	4263	4331	.	+	.	ID=exon:MT-TI-1.1;Parent=transcript:MT-TI-1
NC_012920.1	rCRS	gene	4329	4400	.	-	.	ID=gene:MT-TQ;Name=MT-TQ;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	4329	4400	.	-	.	ID=transcript:MT-TQ-1;Parent=gene:MT-TQ;Name=MT-TQ-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	4329	4400	.	-	.	ID=exon:MT-TQ-1.1;Parent=transcript:MT-TQ-1
NC_012920.1	rCRS	gene	4402	4469	.	+	.	ID=gene:MT-TM;Name=MT-TM;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	4402	4469	.	+	.	ID=transcript:MT-TM-1;Parent=gene:MT-TM;Name=MT-TM-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	4402	4469	.	+	.	ID=exon:MT-TM-1.1;Parent=transcript:MT-TM-1
NC_012920.1	rCRS	gene	4470	5511	.	+	.	ID=gene:MT-ND2;Name=MT-ND2;biotype=protein_coding
NC_012920.1	rCRS	mRNA	4470	5511	.	+	.	ID=transcript:MT-ND2-1;Parent=gene:MT-ND2;Name=MT-ND2-1;biotype=protein_coding
NC_012920.1	rCRS	exon	4470	5511	.	+	.	ID=exon:MT-ND2-1.1;Parent=transcript:MT-ND2-1
NC_012920.1	rCRS	CDS	4470	5511	.	+	0	ID=CDS:MT-ND2-1;Parent=transcript:MT-ND2-1
NC_012920.1	rCRS	gene	5512	5579	.	+	.	ID=gene:MT-TW;Name=MT-TW;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	5512	5579	.	+	.	ID=transcript:MT-TW-1;Parent=gene:MT-TW;Name=MT-TW-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	5512	5579	.	+	.	ID=exon:MT-TW-1.1;Parent=transcript:MT-TW-1
NC_012920.1	rCRS	gene	5587	5655	.	-	.	ID=gene:MT-TA;Name=MT-TA;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	5587	5655	.	-	.	ID=transcript:MT-TA-1;Parent=gene:MT-TA;Name=MT-TA-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	5587	5655	.	-	.	ID=exon:MT-TA-1.1;Parent=transcript:MT-TA-1
NC_012920.1	rCRS	gene	5657	5729	.	-	.	ID=gene:MT-TN;Name=MT-TN;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	5657	5729	.	-	.	ID=transcript:MT-TN-1;Parent=gene:MT-TN;Name=MT-TN-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	5657	5729	.	-	.	ID=exon:MT-TN-1.1;Parent=transcript:MT-TN-1
NC_012920.1	rCRS	gene	5761	5826	.	-	.	ID=gene:MT-TC;Name=MT-TC;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	5761	5826	.	-	.	ID=transcript:MT-TC-1;Parent=gene:MT-TC;Name=MT-TC-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	5761	5826	.	-	.	ID=exon:MT-TC-1.1;Parent=transcript:MT-TC-1
NC_012920.1	rCRS	gene	5826	5891	.	-	.	ID=gene:MT-TY;Name=MT-TY;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	5826	5891	.	-	.	ID=transcript:MT-TY-1;Parent=gene:MT-TY;Name=MT-TY-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	5826	5891	.	-	.	ID=exon:MT-TY-1.1;Parent=transcript:MT-TY-1
NC_012920.1	rCRS	gene	5904	7445	.	+	.	ID=gene:MT-CO1;Name=MT-CO1;biotype=protein_coding
NC_012920.1	rCRS	mRNA	5904	7445	.	+	.	ID=transcript:MT-CO1-1;Parent=gene:MT-CO1;Name=MT-CO1-1;biotype=protein_coding
NC_012920.1	rCRS	exon	5904	7445	.	+	.	ID=exon:MT-CO1-1.1;Parent=transcript:MT-CO1-1
NC_012920.1	rCRS	CDS	5904	7445	.	+	0	ID=CDS:MT-CO1-1;Parent=transcript:MT-CO1-1
NC_012920.1	rCRS	gene	7446	7514	.	-	.	ID=gene:MT-TS1;Name=MT-TS1;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	7446	7514	.	-	.	ID=transcript:MT-TS1-1;Parent=gene:MT-TS1;Name=MT-TS1-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	7446	7514	.	-	.	ID=exon:MT-TS1-1.1;Parent=transcript:MT-TS1-1
NC_012920.1	rCRS	gene	7518	7585	.	+	.	ID=gene:MT-TD;Name=MT-TD;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	7518	7585	.	+	.	ID=transcript:MT-TD-1;Parent=gene:MT-TD;Name=MT-TD-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	7518	7585	.	+	.	ID=exon:MT-TD-1.1;Parent=transcript:MT-TD-1
NC_012920.1	rCRS	gene	7586	8269	.	+	.	ID=gene:MT-CO2;Name=MT-CO2;biotype=protein_coding
NC_012920.1	rCRS	mRNA	7586	8269	.	+	.	ID=transcript:MT-CO2-1;Parent=gene:MT-CO2;Name=MT-CO2-1;biotype=protein_coding
NC_012920.1	rCRS	exon	7586	8269	.	+	.	ID=exon:MT-CO2-1.1;Parent=transcript:MT-CO2-1
NC_012920.1	rCRS	CDS	7586	8269	.	+	0	ID=CDS:MT-CO2-1;Parent=transcript:MT-CO2-1
NC_012920.1	rCRS	gene	8295	8364	.	+	.	ID=gene:MT-TK;Name=MT-TK;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	8295	8364	.	+	.	ID=transcript:MT-TK-1;Parent=gene:MT-TK;Name=MT-TK-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	8295	8364	.	+	.	ID=exon:MT-TK-1.1;Parent=transcript:MT-TK-1
NC_012920.1	rCRS	gene	8366	8572	.	+	.	ID=gene:MT-ATP8;Name=MT-ATP8;biotype=protein_coding
NC_012920.1	rCRS	mRNA	8366	8572	.	+	.	ID=transcript:MT-ATP8-1;Parent=gene:MT-ATP8;Name=MT-ATP8-1;biotype=protein_coding
NC_012920.1	rCRS	exon	8366	8572	.	+	.	ID=exon:MT-ATP8-1.1;Parent=transcript:MT-ATP8-1
NC_012920.1	rCRS	CDS	8366	8572	.	+	0	ID=CDS:MT-ATP8-1;Parent=transcript:MT-ATP8-1
NC_012920.1	rCRS	gene	8527	9207	.	+	.	ID=gene:MT-ATP6;Name=MT-ATP6;biotype=protein_coding
NC_012920.1	rCRS	mRNA	8527	9207	.	+	.	ID=transcript:MT-ATP6-1;Parent=gene:MT-ATP6;Name=MT-ATP6-1;biotype=protein_coding
NC_012920.1	rCRS	exon	8527	9207	.	+	.	ID=exon:MT-ATP6-1.1;Parent=transcript:MT-ATP6-1
NC_012920.1	rCRS	CDS	8527	9207	.	+	0	ID=CDS:MT-ATP6-1;Parent=transcript:MT-ATP6-1
NC_012920.1	rCRS	gene	9207	9990	.	+	.	ID=gene:MT-CO3;Name=MT-CO3;biotype=protein_coding
NC_012920.1	rCRS	mRNA	9207	9990	.	+	.	ID=transcript:MT-CO3-1;Parent=gene:MT-CO3;Name=MT-CO3-1;biotype=protein_coding
NC_012920.1	rCRS	exon	9207	9990	.	+	.	ID=exon:MT-CO3-1.1;Parent=transcript:MT-CO3-1
NC_012920.1	rCRS	CDS	9207	9990	.	+	0	ID=CDS:MT-CO3-1;Parent=transcript:MT-CO3-1
NC_012920.1	rCRS	gene	9991	10058	.	+	.	ID=gene:MT-TG;Name=MT-TG;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	9991	10058	.	+	.	ID=transcript:MT-TG-1;Parent=gene:MT-TG;Name=MT-TG-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	9991	10058	.	+	.	ID=exon:MT-TG-1.1;Parent=transcript:MT-TG-1
NC_012920.1	rCRS	gene	10059	10404	.	+	.	ID=gene:MT-ND3;Name=MT-ND3;biotype=protein_coding
NC_012920.1	rCRS	mRNA	10059	10404	.	+	.	ID=transcript:MT-ND3-1;Parent=gene:MT-ND3;Name=MT-ND3-1;biotype=protein_coding
NC_012920.1	rCRS	exon	10059	10404	.	+	.	ID=exon:MT-ND3-1.1;Parent=transcript:MT-ND3-1
NC_012920.1	rCRS	CDS	10059	10404	.	+	0	ID=CDS:MT-ND3-1;Parent=transcript:MT-ND3-1
NC_012920.1	rCRS	gene	10405	10469	.	+	.	ID=gene:MT-TR;Name=MT-TR;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	10405	10469	.	+	.	ID=transcript:MT-TR-1;Parent=gene:MT-TR;Name=MT-TR-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	10405	10469	.	+	.	ID=exon:MT-TR-1.1;Parent=transcript:MT-TR-1
NC_012920.1	rCRS	gene	10470	10766	.	+	.	ID=gene:MT-ND4L;Name=MT-ND4L;biotype=protein_coding
NC_012920.1	rCRS	mRNA	10470	10766	.	+	.	ID=transcript:MT-ND4L-1;Parent=gene:MT-ND4L;Name=MT-ND4L-1;biotype=protein_coding
NC_012920.1	rCRS	exon	10470	10766	.	+	.	ID=exon:MT-ND4L-1.1;Parent=transcript:MT-ND4L-1
NC_012920.1	rCRS	CDS	10470	10766	.	+	0	ID=CDS:MT-ND4L-1;Parent=transcript:MT-ND4L-1
NC_012920.1	rCRS	gene	10760	12137	.	+	.	ID=gene:MT-ND4;Name=MT-ND4;biotype=protein_coding
NC_012920.1	rCRS	mRNA	10760	12137	.	+	.	ID=transcript:MT-ND4-1;Parent=gene:MT-ND4;Name=MT-ND4-1;biotype=protein_coding
NC_012920.1	rCRS	exon	10760	12137	.	+	.	ID=exon:MT-ND4-1.1;Parent=transcript:MT-ND4-1
NC_012920.1	rCRS	CDS	10760	12137	.	+	0	ID=CDS:MT-ND4-1;Parent=transcript:MT-ND4-1
NC_012920.1	rCRS	gene	12138	12206	.	+	.	ID=gene:MT-TH;Name=MT-TH;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	12138	12206	.	+	.	ID=transcript:MT-TH-1;Parent=gene:MT-TH;Name=MT-TH-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	12138	12206	.	+	.	ID=exon:MT-TH-1.1;Parent=transcript:MT-TH-1
NC_012920.1	rCRS	gene	12207	12265	.	+	.	ID=gene:MT-TS2;Name=MT-TS2;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	12207	12265	.	+	.	ID=transcript:MT-TS2-1;Parent=gene:MT-TS2;Name=MT-TS2-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	12207	12265	.	+	.	ID=exon:MT-TS2-1.1;Parent=transcript:MT-TS2-1
NC_012920.1	rCRS	gene	12266	12336	.	+	.	ID=gene:MT-TL2;Name=MT-TL2;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	12266	12336	.	+	.	ID=transcript:MT-TL2-1;Parent=gene:MT-TL2;Name=MT-TL2-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	12266	12336	.	+	.	ID=exon:MT-TL2-1.1;Parent=transcript:MT-TL2-1
NC_012920.1	rCRS	gene	12337	14148	.	+	.	ID=gene:MT-ND5;Name=MT-ND5;biotype=protein_coding
NC_012920.1	rCRS	mRNA	12337	14148	.	+	.	ID=transcript:MT-ND5-1;Parent=gene:MT-ND5;Name=MT-ND5-1;biotype=protein_coding
NC_012920.1	rCRS	exon	12337	14148	.	+	.	ID=exon:MT-ND5-1.1;Parent=transcript:MT-ND5-1
NC_012920.1	rCRS	CDS	12337	14148	.	+	0	ID=CDS:MT-ND5-1;Parent=transcript:MT-ND5-1
NC_012920.1	rCRS	gene	14149	14673	.	-	.	ID=gene:MT-ND6;Name=MT-ND6;biotype=protein_coding
NC_012920.1	rCRS	mRNA	14149	14673	.	-	.	ID=transcript:MT-ND6-1;Parent=gene:MT-ND6;Name=MT-ND6-1;biotype=protein_coding
NC_012920.1	rCRS	exon	14149	14673	.	-	.	ID=exon:MT-ND6-1.1;Parent=transcript:MT-ND6-1
NC_012920.1	rCRS	CDS	14149	14673	.	-	0	ID=CDS:MT-ND6-1;Parent=transcript:MT-ND6-1
NC_012920.1	rCRS	gene	14674	14742	.	-	.	ID=gene:MT-TE;Name=MT-TE;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	14674	14742	.	-	.	ID=transcript:MT-TE-1;Parent=gene:MT-TE;Name=MT-TE-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	14674	14742	.	-	.	ID=exon:MT-TE-1.1;Parent=transcript:MT-TE-1
NC_012920.1	rCRS	gene	14747	15887	.	+	.	ID=gene:MT-CYB;Name=MT-CYB;biotype=protein_coding
NC_012920.1	rCRS	mRNA	14747	15887	.	+	.	ID=transcript:MT-CYB-1;Parent=gene:MT-CYB;Name=MT-CYB-1;biotype=protein_coding
NC_012920.1	rCRS	exon	14747	15887	.	+	.	ID=exon:MT-CYB-1.1;Parent=transcript:MT-CYB-1
NC_012920.1	rCRS	CDS	14747	15887	.	+	0	ID=CDS:MT-CYB-1;Parent=transcript:MT-CYB-1
NC_012920.1	rCRS	gene	15888	15953	.	+	.	ID=gene:MT-TT;Name=MT-TT;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	15888	15953	.	+	.	ID=transcript:MT-TT-1;Parent=gene:MT-TT;Name=MT-TT-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	15888	15953	.	+	.	ID=exon:MT-TT-1.1;Parent=transcript:MT-TT-1
NC_012920.1	rCRS	gene	15956	16023	.	-	.	ID=gene:MT-TP;Name=MT-TP;biotype=Mt_tRNA
NC_012920.1	rCRS	mRNA	15956	16023	.	-	.	ID=transcript:MT-TP-1;Parent=gene:MT-TP;Name=MT-TP-1;biotype=Mt_tRNA
NC_012920.1	rCRS	exon	15956	16023	.	-	.	ID=exon:MT-TP-1.1;Parent=transcript:MT-TP-1
EOF

wc -l chrM.gff3
head -8 chrM.gff3

`bcftools csq` trabaja con el GFF3 comprimido e indexado, igual que hicimos con el VCF de sitios conocidos en la clase 5:

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase7
bgzip -f chrM.gff3
tabix -p gff chrM.gff3.gz

Ahora sí, anotamos (`-p a`: como nuestras variantes no vienen "en fase" entre sí, le pedimos que las trate como haplotipos independientes, que es lo que queremos para anotar cada variante por separado):

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase7
REF=~/bioinfo/clase5/referencia/chrM.fasta

bcftools csq -f "$REF" -g chrM.gff3.gz -p a resultados/mtDNA_sample.vcf \
    -Ov -o resultados/mtDNA_sample.anotado.vcf

La anotación queda en el campo `INFO/BCSQ`, como texto separado por `|`. Miremos solo eso, por variante:

In [ ]:
%%bash
source "$HOME/miniforge3/etc/profile.d/conda.sh"
conda activate mejoramiento

cd ~/bioinfo/clase7
bcftools query -f '%POS\t%REF\t%ALT\t%INFO/BCSQ\n' resultados/mtDNA_sample.anotado.vcf

El formato de `BCSQ` es `tipo_de_consecuencia|gen|transcrito|biotipo|hebra|cambio_de_aminoácido|cambio_de_nucleótido`. Algunos ejemplos de su propio resultado:

- **Posición 591** → `non_coding|MT-TF|...` : cae dentro de `MT-TF`, un gen de ARN de transferencia. Como no codifica proteína, no hay "cambio de aminoácido" que evaluar —el efecto de esta variante hay que pensarlo a nivel de la estructura del ARNt, no de una proteína—.
- **Posición 3875** → `missense|MT-ND1|...|190L>190Q|...` : en el codón 190 del gen `MT-ND1`, el aminoácido cambia de Leucina (L) a Glutamina (Q). Una variante **missense** cambia el aminoácido; puede o no afectar la función de la proteína, depende de qué tan importante sea esa posición.
- **Posición 4756** → `stop_gained|MT-ND2|...|96L>96*` : en el codón 96 de `MT-ND2`, el cambio crea un codón de término (`*`) donde antes había Leucina. Esto trunca la proteína a partir de ahí —en general, una consecuencia más severa que un missense—.

> **Limitación importante:** `bcftools csq` traduce usando el **código genético estándar** (el mismo de un gen nuclear). El genoma mitocondrial humano usa un código **ligeramente distinto**: por ejemplo, el codón `ATA` es Metionina en mitocondria (no Isoleucina, como en el código estándar). Revisen la posición **11529** en su resultado: el codón de referencia ahí es justo `ATA`, así que el aminoácido de referencia que reporta `bcftools csq` (Isoleucina) no es el correcto para mitocondria (debería ser Metionina) —el *tipo* de consecuencia (missense) sigue siendo válido, pero el aminoácido específico no—. Para un análisis real de variantes mitocondriales se usan herramientas que sí conocen este código (por ejemplo MITOMAP, MToolBox); `bcftools csq` nos sirve hoy para practicar la mecánica de anotar, no para un reporte clínico.

---
## 8. Interpretación: ¿cuáles variantes miramos primero? (15 min)

Con 11 variantes y sus anotaciones, ¿por dónde empezar? Una forma simple de priorizar, de más a menos probable que tenga un efecto biológico:

1. **`stop_gained` / `frameshift`** en un gen que codifica proteína: trunca o desordena la proteína completa — el tipo de cambio más severo.
2. **`missense`** en un gen que codifica proteína: cambia un aminoácido — el efecto depende de la posición y de qué tan distinto es el aminoácido nuevo.
3. **`synonymous`**: cambia el codón pero no el aminoácido — generalmente sin efecto en la proteína (aunque no siempre: puede afectar el splicing o la estabilidad del ARN, algo que no vamos a profundizar hoy).
4. **`non_coding`** (en un ARNt o ARNr): no hay aminoácido que evaluar, pero el ARN igual puede verse afectado si el cambio altera su estructura.

Esto es solo un punto de partida: para saber si una variante concreta **ya se describió** como relevante (por ejemplo, asociada a una enfermedad mitocondrial), el siguiente paso en un análisis real sería buscarla en una base de datos curada como **MITOMAP** o **ClinVar** — no algo que nuestro anotador simplificado pueda decirnos.

> **Ejercicio rápido (en parejas, 5 min):** de las 11 variantes anotadas, identifiquen las que son `stop_gained`. Para cada una, escriban en una frase qué esperarían que le pase a la proteína correspondiente.

---
## 9. Ejercicio integrador (en parejas, 25 min)

Repitan todo el flujo de hoy sobre la muestra que generaron ustedes en el ejercicio de la clase 6 (`mtDNA_pareja`).

1. Verifiquen que tienen `~/bioinfo/clase6/resultados/mtDNA_pareja.final.bam` (si no, córranlo primero: `~/bioinfo/clase6/mejorar_bam.sh` con los parámetros de su muestra).
2. Llamen variantes con `gatk HaplotypeCaller`, guardando el resultado en `~/bioinfo/clase7/resultados/mtDNA_pareja.vcf`.
3. Comparen contra su respuesta conocida (`~/bioinfo/clase6/sitios_conocidos_pareja.vcf` o el nombre que le hayan puesto), igual que en la sección 6.
4. Anoten con `bcftools csq`, usando el mismo `chrM.gff3.gz` de hoy (no hace falta rehacerlo, las coordenadas de los genes no cambian entre muestras).
5. Elijan **una** variante `missense` o `stop_gained` de su muestra y expliquen, en sus palabras: ¿en qué gen cae?, ¿qué aminoácido cambia por cuál?, ¿la clasificarían como algo a mirar con más atención, según la priorización de la sección 8?
6. **Reporten:** ¿cuántas variantes encontraron?, ¿coincide con la respuesta conocida?, ¿se les escapó alguna (y por qué, si pueden explicarlo)?

No hay una única forma correcta de resolverlo. Lo importante es que puedan explicar cada paso y cada anotación con sus propias palabras.

---
## 10. Cierre y resumen

### Comandos vistos hoy

```
Instalacion:   conda install -n mejoramiento -c bioconda bcftools
Llamado:       gatk HaplotypeCaller -R ref.fasta -I muestra.final.bam -O muestra.vcf
Comprimir:     bgzip archivo.gff3 ; tabix -p gff archivo.gff3.gz
Anotacion:     bcftools csq -f ref.fasta -g anotacion.gff3.gz -p a muestra.vcf -Ov -o muestra.anotado.vcf
Ver anotacion: bcftools query -f '%POS\t%REF\t%ALT\t%INFO/BCSQ\n' muestra.anotado.vcf
Comparar:      grep -v '^#' archivo.vcf | cut -f2 | sort -n > posiciones.txt
               grep -Fxf posiciones_A.txt posiciones_B.txt        (comunes)
               grep -Fxvf posiciones_A.txt posiciones_B.txt       (solo en B)
```

### Ideas clave

- Un VCF tiene 8 columnas obligatorias (`CHROM` a `INFO`) más `FORMAT` y una columna por muestra; el genotipo (`GT`) dice qué alelo(s) tiene la muestra, y `AD`/`DP`/`GQ` dicen qué tan bien apoyado está ese genotipo por las lecturas.
- Anotar una variante es ponerla en contexto: ¿en qué gen cae?, ¿cambia un aminoácido?, ¿qué tan severo es ese cambio?
- Ninguna herramienta es perfecta para todos los casos: `bcftools csq` usa el código genético estándar, que no es exactamente el del mtDNA — parte de interpretar bien un resultado es saber qué asume la herramienta que lo generó.

### Lo que dejamos listo para la próxima clase

- `~/bioinfo/clase7/resultados/mtDNA_sample.vcf` y `mtDNA_sample.anotado.vcf` (y su equivalente para `mtDNA_pareja`).
- `~/bioinfo/clase7/chrM.gff3.gz` (+ su índice `.tbi`): la anotación de genes mitocondriales, reusable para cualquier muestra futura de mtDNA.

### Tarea para antes de la próxima clase

1. Terminen el ejercicio integrador si no alcanzaron.
2. Para las 11 variantes de `mtDNA_sample`, anoten en una tabla propia (papel o spreadsheet): posición, gen, tipo de consecuencia, y si la priorizarían o no según la sección 8.
3. Repasen las columnas del VCF y los campos `GT`/`AD`/`DP`/`GQ` (sección 5): la próxima clase los vamos a dar por conocidos.